# 05 — Field profile validation: 1+1D vs 3+1D  (lb=0.84)

Compares φ(z, t) directly between BubbleMaster (1+1D) and sledgehamr (3+1D)
across all available snapshot times.

**Workflow**
1. Run **Section 1** to list available sledgehamr snapshot times.
2. Run **Section 2** to write BubbleMaster setup files and SLURM scripts.
3. Submit: `! sbatch scripts/fields_<tag>.sh`
4. After jobs complete, run **Sections 3–5** to load, compare, and quantify the time offset.

In [ ]:
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path

import ptbuilder as pt
from ptbuilder.ic import _cutoff_times, BubbleMasterParams

REPO_ROOT = Path(pt.__file__).parent.parent

PYSLEDGEHAMR_PATH = '/global/cfs/cdirs/m3166/buschman/sledgehamr'
sys.path.append(PYSLEDGEHAMR_PATH)
import pySledgehamr as sledgehamr

config = pt.Config()
params = BubbleMasterParams(dz=0.005, n_w=64, n_k=51,
                             how_often_ds=5, baby_steps=100, cutoff_type=0)

sh_base = '/pscratch/sd/b/buschman/other_runs/'

# lb=0.84: bubble_N3_234_0_1_  → bubbles 0=(72.25,89.92,100) & 1=(127.8,89.92,100)
#           Δ=(55.55, 0, 0) — x-aligned, midpoint x=(72.25+127.8)/2=100.025
# GetSlice(axis='x') is fixed at HDF5 zloc=L/2=100 which equals the bubble z-centre,
# so the slice passes exactly through the equatorial plane — no geometric correction needed.
VALIDATION_POINTS = [
    dict(lambda_bar=0.84, d=55.55,
         sh_path=sh_base+'bubble_N3_234_0_1_',
         sh_N0=2048, sh_L=200., sh_axis='x', sh_mid_x=100.025, sh_mid_y=89.92, sh_lev=0),
]

def gamma_for(lambda_bar, d):
    model = pt.PhysicsModel(pt.Phi4Potential(lambda_bar), config)
    return float(model.kinematics.Gamma(model.kinematics.collision_time(d)))

def vp_tag(vp):
    return f'lb{vp["lambda_bar"]}_g{vp["gamma"]:.4g}'

for vp in VALIDATION_POINTS:
    vp['gamma'] = gamma_for(vp['lambda_bar'], vp['d'])
    t_0, t_cut, t_m, _ = _cutoff_times(vp['d'], params.cutoff_type, params.t_0_scal)
    model   = pt.PhysicsModel(pt.Phi4Potential(vp['lambda_bar']), config)
    t_coll  = float(model.kinematics.collision_time(vp['d']))
    vp['t_coll'] = t_coll
    vp['t_m']    = t_m
    print(f'{vp_tag(vp)}  d={vp["d"]}  t_coll={t_coll:.3f}  t_m={t_m:.3f}')

## 1. List sledgehamr snapshot times

In [ ]:
for vp in VALIDATION_POINTS:
    output  = sledgehamr.Output(vp['sh_path'])
    t_snaps = output.GetTimesOfSlices()
    print(f"{vp_tag(vp)}  {Path(vp['sh_path']).name}  ({len(t_snaps)} slices)")
    for i, t in enumerate(t_snaps):
        print(f'  [{i:3d}]  t = {t:.4f}')
    print()

## 2. Prerequisites

Run notebook 04 first — it writes the SLURM scripts and submits the BubbleMaster jobs
(with `--save-fields`) that produce the `fields.h5` files loaded below.
The output lives in `validation/<tag>/out/`, shared with the spectrum comparison.

## 3. Helper functions

In [ ]:
def load_bm_fields(out_dir):
    """Return (s, z, phi) from fields.h5; phi has shape (n_s, n_z)."""
    with h5py.File(Path(out_dir) / 'fields.h5', 'r') as f:
        return f['s'][:], f['z'][:], f['phi'][:]


def extract_bm_band(s_bm, z_bm, phi_bm, t_snap):
    """
    Return BM field profiles at the two s-snapshots bracketing t_snap.

    s is Minkowski time, z is the Minkowski spatial coordinate along the
    collision axis — no coordinate transformation is needed.  The band
    between the two bracketing snapshots shows the BM time resolution.
    """
    i_lo = np.clip(np.searchsorted(s_bm, t_snap, side='right') - 1,
                   0, len(s_bm) - 2)
    i_hi = i_lo + 1
    return phi_bm[i_lo, :], phi_bm[i_hi, :], s_bm[i_lo], s_bm[i_hi]


def get_sh_field_1d(output, snap_idx, L, mid_x, mid_y, axis='x', lev=0):
    """
    1D profile along `axis` at the transverse midpoint row.
    Derives grid size from the actual returned array — no N0 needed.
    Returns (z_sh, phi_sh, t) with z_sh=0 at the collision midpoint.
    """
    slices  = output.GetSlice(snap_idx, axis, lev, ['Phi'])
    t       = slices['t']
    phi_2d  = slices['Phi']
    n_rows, n_cols = phi_2d.shape
    row     = int(round(mid_y * n_rows / L))
    row     = max(0, min(row, n_rows - 1))
    phi_1d  = phi_2d[row, :]
    z_sh    = np.linspace(0, L, n_cols) - mid_x
    return z_sh, phi_1d, t


def closest_snap(t_snaps, t_target):
    return int(np.argmin(np.abs(t_snaps - t_target)))

## 3. Load data

In [ ]:
# --- BubbleMaster fields ---
bm_data = {}
for vp in VALIDATION_POINTS:
    tag     = vp_tag(vp)
    model   = pt.PhysicsModel(pt.Phi4Potential(vp['lambda_bar']), config)
    out_dir = model.results_dir / 'validation' / tag / 'out'
    try:
        s, z, phi = load_bm_fields(out_dir)
        bm_data[tag] = dict(s=s, z=z, phi=phi)
        print(f'[{tag}]  s=[{s[0]:.3f},{s[-1]:.3f}]  z=[{z[0]:.3f},{z[-1]:.3f}]  phi shape={phi.shape}')
    except FileNotFoundError:
        print(f'[{tag}]  fields.h5 not found — run notebook 04 first')

# --- All available sledgehamr slices ---
sh_data = {}
for vp in VALIDATION_POINTS:
    tag     = vp_tag(vp)
    output  = sledgehamr.Output(vp['sh_path'])
    t_snaps = output.GetTimesOfSlices()
    snaps   = []
    n_fail  = 0
    for idx in range(len(t_snaps)):
        try:
            z_sh, phi_sh, t_sh = get_sh_field_1d(
                output, idx, vp['sh_L'], vp['sh_mid_x'], vp['sh_mid_y'],
                axis=vp['sh_axis'], lev=vp['sh_lev'])
            snaps.append(dict(idx=idx, t=float(t_sh), z=z_sh, phi=phi_sh))
        except Exception:
            n_fail += 1
    if tag in bm_data:
        s_max = bm_data[tag]['s'][-1]
        snaps = [sn for sn in snaps if sn['t'] <= s_max]
    sh_data[tag] = snaps
    if snaps:
        print(f'[{tag}]  {len(snaps)} slices loaded, {n_fail} failed  '
              f't=[{snaps[0]["t"]:.3f}, {snaps[-1]["t"]:.3f}]')
    else:
        print(f'[{tag}]  no slices loaded ({n_fail} failed)')

## 4. Plot

In [ ]:
N_COLS = 4   # panels per row

for vp in VALIDATION_POINTS:
    tag    = vp_tag(vp)
    color  = '#0072B2'
    d      = vp['d']
    bm     = bm_data.get(tag)
    snaps  = sh_data.get(tag, [])
    if not snaps:
        print(f'[{tag}]  no 3D slices')
        continue

    n      = len(snaps)
    n_cols = min(N_COLS, n)
    n_rows = (n + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(4.5 * n_cols, 3.2 * n_rows),
                              squeeze=False)
    fig.suptitle(rf'$\bar\lambda={vp["lambda_bar"]}$  raw comparison  '
                 r'(1+1D at same $t$, no shift)', fontsize=10)

    for i, snap in enumerate(snaps):
        ax     = axes[i // n_cols][i % n_cols]
        t_snap = snap['t']
        mask   = snap['z'] >= 0

        ax.plot(snap['z'][mask], snap['phi'][mask], color='black', lw=1.5, label='3+1D', ls=':')

        if bm is not None and t_snap <= bm['s'][-1]:
            phi_lo, phi_hi, _, _ = extract_bm_band(bm['s'], bm['z'], bm['phi'], t_snap)
            ax.fill_between(bm['z'], phi_lo, phi_hi,
                            color='darkorange', alpha=1, linewidth=1, label='1+1D')

        ax.set_xlim(0, d * 0.7)
        ax.set_title(f't = {t_snap:.3f}', fontsize=8)
        ax.set_xlabel('z', fontsize=7)
        ax.set_ylabel(r'$\phi$', fontsize=7)
        ax.tick_params(labelsize=6)
        if i == 0:
            ax.legend(frameon=False, fontsize=7)

    # Hide unused panels
    for j in range(n, n_rows * n_cols):
        axes[j // n_cols][j % n_cols].set_visible(False)

    plt.tight_layout()
    plt.savefig(REPO_ROOT / f'validate_fields_{tag}.pdf', bbox_inches='tight')
    plt.show()

## 5. Time offset between 3D and 1D profiles (lb=0.84)

For each 3D snapshot find the BM time `t_BM` that minimises the L² profile residual
`||φ_BM(z, t_BM) − φ_3D(z)||²` over the overlapping z-domain.
The offset `Δt = t_3D − t_BM` tells us how much earlier (or later) the 3D bubble
is relative to the 1D solution at the same profile shape.

In [ ]:
from scipy.optimize import minimize_scalar
from scipy.interpolate import RegularGridInterpolator

results_offset = {}   # tag → list of dicts

for vp in VALIDATION_POINTS:
    tag = vp_tag(vp)
    bm  = bm_data.get(tag)
    sh  = sh_data.get(tag, [])
    if bm is None or not sh:
        print(f'[{tag}]  data missing — skip')
        continue

    s_bm, z_bm, phi_bm = bm['s'], bm['z'], bm['phi']

    bm_interp = RegularGridInterpolator(
        (s_bm, z_bm), phi_bm,
        method='linear', bounds_error=False, fill_value=0.)

    offsets = []
    print(f'[{tag}]')
    print(f'  {"idx":>4}  {"t_3D":>8}  {"t_BM_opt":>10}  {"Δt":>8}  {"residual":>10}')

    for snap in sh:
        t_3D   = snap['t']
        z_sh   = snap['z']
        phi_sh = snap['phi']

        z_lo     = max(z_sh.min(), z_bm.min())
        z_hi     = min(z_sh.max(), z_bm.max())
        z_common = z_sh[(z_sh >= z_lo) & (z_sh <= z_hi)]
        phi_sh_c = phi_sh[(z_sh >= z_lo) & (z_sh <= z_hi)]

        if len(z_common) < 4:
            continue

        def residual(t_bm):
            pts = np.column_stack([np.full(len(z_common), t_bm), z_common])
            return float(np.mean((bm_interp(pts) - phi_sh_c)**2))

        res      = minimize_scalar(residual,
                                   bounds=(s_bm[0], min(t_3D * 1.5, s_bm[-1])),
                                   method='bounded', options=dict(xatol=1e-5))
        t_bm_opt = res.x
        dt       = t_3D - t_bm_opt
        offsets.append(dict(idx=snap['idx'], t_3D=t_3D, t_bm=t_bm_opt, dt=dt, resid=res.fun))
        print(f'  {snap["idx"]:>4}  {t_3D:>8.4f}  {t_bm_opt:>10.4f}  {dt:>+8.4f}  {res.fun:>10.2e}')

    results_offset[tag] = offsets
    dts = [o['dt'] for o in offsets]
    if dts:
        print(f'  → mean Δt = {np.mean(dts):+.4f}   std = {np.std(dts):.4f}   '
              f'median = {np.median(dts):+.4f}')

# Δt vs t plot
fig, ax = plt.subplots(figsize=(6, 3.5))
for vp in VALIDATION_POINTS:
    tag = vp_tag(vp)
    offs = results_offset.get(tag, [])
    if not offs:
        continue
    t_vals  = np.array([o['t_3D'] for o in offs])
    dt_vals = np.array([o['dt']   for o in offs])
    ax.plot(t_vals, dt_vals, 'o-', color='#0072B2', markersize=5, label=tag)
    ax.axhline(np.mean(dt_vals), color='#0072B2', ls='--', lw=0.8,
               label=f'mean Δt = {np.mean(dt_vals):+.4f}')
    ax.axvline(vp['t_coll'], color='grey', lw=0.8, ls=':',
               label=f't_coll = {vp["t_coll"]:.3f}')
ax.set_xlabel('$t_{3D}$')
ax.set_ylabel(r'$\Delta t = t_\mathrm{3D} - t_\mathrm{BM,opt}$')
ax.set_title('Profile-matched time offset', fontsize=9)
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig(REPO_ROOT / 'validate_fields_dt.pdf', bbox_inches='tight')
plt.show()

### Time-shifted profile comparison

For each 3D snapshot, the 1+1D profile is shown at the BM time `t_BM_opt` that minimises
the profile residual — i.e. at the optimal time shift found above.

In [ ]:
for vp in VALIDATION_POINTS:
    tag    = vp_tag(vp)
    color  = '#0072B2'
    d      = vp['d']
    bm     = bm_data.get(tag)
    snaps  = sh_data.get(tag, [])
    offs   = results_offset.get(tag, [])
    if not snaps or bm is None or not offs:
        continue

    # Build a lookup: snap idx → optimal BM time
    opt_t = {o['idx']: o['t_bm'] for o in offs}

    s_bm, z_bm, phi_bm = bm['s'], bm['z'], bm['phi']

    n      = len(snaps)
    n_cols = min(N_COLS, n)
    n_rows = (n + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(4.5 * n_cols, 3.2 * n_rows),
                              squeeze=False)
    fig.suptitle(rf'$\bar\lambda={vp["lambda_bar"]}$  time-shifted comparison  '
                 r'(1+1D at optimal $t_\mathrm{BM}$)', fontsize=10)

    for i, snap in enumerate(snaps):
        ax      = axes[i // n_cols][i % n_cols]
        t_3D    = snap['t']
        mask_3d = snap['z'] >= 0

        ax.plot(snap['z'][mask_3d], snap['phi'][mask_3d],
                color='black', lw=1.5, label='3+1D', ls=':')

        t_bm_opt = opt_t.get(snap['idx'])
        if t_bm_opt is not None and s_bm[0] <= t_bm_opt <= s_bm[-1]:
            phi_lo, phi_hi, _, _ = extract_bm_band(s_bm, z_bm, phi_bm, t_bm_opt)
            ax.fill_between(z_bm, phi_lo, phi_hi,
                            color='darkorange', alpha=1, linewidth=1,
                            label=f'1+1D  $t_{{BM}}$={t_bm_opt:.3f}')
            dt = t_3D - t_bm_opt
            ax.set_title(f't$_{{3D}}$={t_3D:.3f}   Δt={dt:+.4f}', fontsize=7.5)
        else:
            ax.set_title(f't$_{{3D}}$={t_3D:.3f}  (no shift)', fontsize=7.5)

        ax.set_xlim(0, d * 0.7)
        ax.set_xlabel('z', fontsize=7)
        ax.set_ylabel(r'$\phi$', fontsize=7)
        ax.tick_params(labelsize=6)
        if i == 0:
            ax.legend(frameon=False, fontsize=7)

    for j in range(n, n_rows * n_cols):
        axes[j // n_cols][j % n_cols].set_visible(False)

    plt.tight_layout()
    plt.savefig(REPO_ROOT / f'validate_fields_shifted_{tag}.pdf', bbox_inches='tight')
    plt.show()